# GREEN metrics

**GREEN** (Generative Radiology Report Evaluation and Error Notation) is the most advanced metric in the field of Radiology Report Generation (RRG) to date, developed by Stanford researchers (Stanford AIMI) in late 2024.

It operates on the **LLM-as-a-judge** principle. GREEN uses a specialized language model (e.g., `StanfordAIMI/GREEN-radllama-7b-v1`) that fully simulates the work of a human radiologist: it doesn't simply count word or graph matches, but instead detects, classifies, and provides textual explanations for AI errors (hallucinations, missed diagnoses, incorrect anatomy).

To install GREEN metrics:

    # Create a virtual environment for Python 3.12.1, which is required for GREEN
    conda create -n med_metrics python=3.12.1
    conda activate med_metrics

    # Install GREEN metrics and all its dependences
    pip install green-score bitsandbytes accelerate transformers>=4.38.0

    # Install additional libraries
    pip install pandas numpy nltk tqdm jupyter

**Warning**: for now, runtime on `Tesla V100-PCIE-16GB` GPU is **17 hours**.

In [1]:
GPU = 3  # GPU number. After change, restart the kernel

%set_env CUDA_VISIBLE_DEVICES={GPU}

import os
import torch

if torch.cuda.is_available():  # make sure GPU is available
    num = torch.cuda.device_count()
    print(f"GPU count: {num}")
    for i in range(num):
        print(f"GPU {i} name: {torch.cuda.get_device_name(i)}")
else:
    print("GPU is not available")


# Disable tokenizer multi-fork warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"

env: CUDA_VISIBLE_DEVICES=3
GPU count: 1
GPU 0 name: Tesla V100-PCIE-16GB


In [2]:
import os
import sys
import torch
import warnings
import pandas as pd

from tqdm.notebook import tqdm
from green_score import GREEN
import green_score.green as gs_module  # Safe top-level module link

# Suppress all background console logging pollution
warnings.filterwarnings("ignore", category=FutureWarning, module="huggingface_hub")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=".*Number of distinct clusters.*")

# Prevent PyTorch VRAM fragmentation on Tesla V100
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


def evaluate_all_columns(csv_file):
    """
    Evaluates metrics natively at maximum speed within a 16GB VRAM ceiling.
    """
    df = pd.read_csv(csv_file)

    # Random subsampling (4x speedup)
    # We take 1200 random rows. random_state=42 guarantees reproducibility across runs.
    if len(df) > 1200:
        df = df.sample(n=1200, random_state=42).reset_index(drop=True)
        print(f"Dataset successfully subsampled to {len(df)} rows for acceleration.")
    else:
        print(f"Dataset has {len(df)} rows, skipping subsampling.")

    actual_reports = df["actual_text"].tolist()

    # Hardcoded columns list
    columns_list = [
        "test_finetuned_qwen_v05",
        "test_finetuned_qwen_v06",
        "best_model_lstm_5_layers_resnet_unfreeze_top_p=0.1",
        "best_model_lstm_5_layers_resnet_unfreeze_top_p=0.4",
        "best_model_lstm_5_layers_resnet_unfreeze_top_p=0.5",
        # "test_base_qwen",
        "test_finetuned_qwen_v01",
        "test_finetuned_qwen_v02",
        "test_finetuned_qwen_v03",
        "test_finetuned_qwen_v04",
        "test_vit-gpt2_transformer_v01",
        "test_vit-gpt2_transformer_v02",
        "test_vit-gpt2_transformer_v03",
        "test_vit-gpt2_transformer_v04",
        "test_vit-gpt2_transformer_v05",
        "test_vit-gpt2_transformer_v06",
        "test_vit-gpt2_transformer_v07",
    ]

    # Double-check safety to process only columns that strictly exist in the CSV file
    columns_list = [m for m in columns_list if m in df.columns]
    print(*columns_list, sep="\n")

    print("=== Loading High-Speed Stanford GREEN (RadPhi2) Evaluator ===")

    # Silence initial Hugging Face download output
    old_stdout = sys.stdout
    sys.stdout = open(os.devnull, "w")

    try:
        torch.set_default_dtype(torch.float16)
        
        # SDPA ATTENTION PATCH (Forced hardware acceleration backend)
        import transformers
        old_from_pretrained = transformers.AutoModelForCausalLM.from_pretrained
        
        def fast_from_pretrained(cls, model_id, *args, **kwargs):
            kwargs["attn_implementation"] = "sdpa"
            kwargs["device_map"] = "auto"
            return old_from_pretrained(model_id, *args, **kwargs)

        transformers.AutoModelForCausalLM.from_pretrained = classmethod(fast_from_pretrained)

        evaluator = GREEN(model_name="StanfordAIMI/GREEN-RadPhi2")
        evaluator.batch_size = 6  # Static V100 ceiling sweet-spot

        # Discard CPu clustring and text export refactor
        gs_module.GREEN.get_summary = lambda *args, **kwargs: "Disabled"

    finally:
        sys.stdout = old_stdout

    print("=== Starting Fast Stanford GREEN Evaluation Pipeline ===")
    summary_data = []

    # Iterate cleanly over target column slices
    for col in tqdm(columns_list, desc="Overall Column Progress"):
        predicted_reports = df[col].tolist()
        torch.cuda.empty_cache()

        # In-place progress bar tracking
        gs_module.tqdm_on_main = lambda iterable, total, **kwargs: tqdm(
            iterable, total=total, desc=f"Evaluating {col}", leave=False
        )

        # Run inference safely without chaining risky direct indexing
        outputs = evaluator(hyps=predicted_reports, refs=actual_reports)

        # Safely capture the score without any risky index bracketing
        mean_score = float(outputs[0]) if isinstance(outputs, (tuple, list)) else float(outputs)

        summary_data.append({
            "Model_Column": col,
            "GREEN_Score": float(mean_score)
        })

        print(f"\t{col}: {mean_score:.3f} <-- higher is better\n")

    # Save minimal leaderboard summary
    pd.DataFrame(summary_data).to_csv("green_metrics_summary.csv", index=False)
    print("=== Pipeline Complete! Final scores saved to 'green_metrics_summary.csv' ===")


if __name__ == "__main__":
    evaluate_all_columns("../test_all.csv")

Dataset successfully subsampled to 1200 rows for acceleration.
test_finetuned_qwen_v05
test_finetuned_qwen_v06
best_model_lstm_5_layers_resnet_unfreeze_top_p=0.1
best_model_lstm_5_layers_resnet_unfreeze_top_p=0.4
best_model_lstm_5_layers_resnet_unfreeze_top_p=0.5
test_finetuned_qwen_v01
test_finetuned_qwen_v02
test_finetuned_qwen_v03
test_finetuned_qwen_v04
test_vit-gpt2_transformer_v01
test_vit-gpt2_transformer_v02
test_vit-gpt2_transformer_v03
test_vit-gpt2_transformer_v04
test_vit-gpt2_transformer_v05
test_vit-gpt2_transformer_v06
test_vit-gpt2_transformer_v07
=== Loading High-Speed Stanford GREEN (RadPhi2) Evaluator ===


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

=== Starting Fast Stanford GREEN Evaluation Pipeline ===


Overall Column Progress:   0%|          | 0/16 [00:00<?, ?it/s]

Processing data...making prompts


...

==== End Inference ====
Computing summary ...
Seconds per example:  3.5510773040850956
	test_vit-gpt2_transformer_v06: 0.090 <-- higher is better

Processing data...making prompts


Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Done.


Evaluating test_vit-gpt2_transformer_v07:   0%|          | 0/200 [00:00<?, ?it/s]

==== End Inference ====
Computing summary ...
Seconds per example:  3.6206273474295934
	test_vit-gpt2_transformer_v07: 0.146 <-- higher is better

=== Pipeline Complete! Final scores saved to 'green_metrics_summary.csv' ===
